In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ==========================================

file_name = '/content/sample_data/dataset.csv'
# ==========================================

# 1. Load Dataset
try:
    df = pd.read_csv(file_name)
    print("==================================================")
    print(" STEP 1: DATASET INITIAL STATE")
    print("==================================================")
    print(f"🔹 Original Rows & Columns (Shape): {df.shape}")
    print("\n🔹 Pehle 3 Rows (Original Data):")
    print(df.head(3))
except FileNotFoundError:
    print(f" Error: '{file_name}' nahi mili. File upload check karein.")
    df = None

if df is not None:
    # Handle Target & IDs if missing (For demo safety)
    if 'target' not in df.columns: df['target'] = np.random.choice([0, 1], size=len(df))
    if 'user_id' not in df.columns: df['user_id'] = np.random.randint(1, 100, size=len(df))
    if 'track_id' not in df.columns: df['track_id'] = df.index

    print("\n==================================================")
    print(" STEP 2: CLEANING & MISSING VALUES")
    print("==================================================")
    print(" Missing values per column before cleaning:")
    print(df.isnull().sum().filter(items=['tempo', 'loudness', 'genre', 'track_genre', 'artist_name', 'artists']))

    # Drop rows with missing values
    df = df.dropna()
    print(f"\n🔹 Shape after dropping missing values: {df.shape}")

    print("\n==================================================")
    print("STEP 3: CATEGORICAL ENCODING (Genre / Artist)")
    print("==================================================")
    categorical_cols = [col for col in ['genre', 'track_genre', 'artist_name', 'artists'] if col in df.columns]

    if categorical_cols:
        print(f"🔹 Encoding columns: {categorical_cols}")
        for col in categorical_cols:
            print(f"\n'{col}' ke pehle 3 original values: {df[col].head(3).tolist()}")
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            print(f" '{col}' ke pehle 3 encoded (numerical) values: {df[col].head(3).tolist()}")
    else:
        print(" Koi categorical column nahi mila.")

    print("\n==================================================")
    print(" STEP 4: NORMALIZATION (Tempo & Loudness)")
    print("==================================================")
    continuous_cols = [col for col in ['tempo', 'loudness', 'danceability', 'energy', 'valence', 'popularity'] if col in df.columns]

    if continuous_cols:
        print("🔹 Normalization se pehle stats (Mean & Std):")
        print(df[continuous_cols].describe().loc[['mean', 'std']])

        # Scaling
        scaler = StandardScaler()
        df[continuous_cols] = scaler.fit_transform(df[continuous_cols])

        print("\n🔹 Normalization ke baad stats (Mean should be ~0, Std should be ~1):")
        print(df[continuous_cols].describe().loc[['mean', 'std']].round(2))
    else:
        print(" Koi continuous column nahi mila.")

    print("\n==================================================")
    print(" FINAL PROCESSED DATA (First 5 Rows)")
    print("==================================================")
    # Target, features aur IDs ko aik sath show karna
    display_cols = ['user_id', 'track_id'] + continuous_cols + categorical_cols + ['target']
    print(df[display_cols].head())

 STEP 1: DATASET INITIAL STATE
🔹 Original Rows & Columns (Shape): (114000, 21)

🔹 Pehle 3 Rows (Original Data):
   Unnamed: 0                track_id                 artists  \
0           0  5SuOikwiRyPMVoIQDJUgSV             Gen Hoshino   
1           1  4qPNDBW1i3p13qLCt0Ki3A            Ben Woodward   
2           2  1iJBSr7s7jYXzM8EGcbK5b  Ingrid Michaelson;ZAYN   

         album_name        track_name  popularity  duration_ms  explicit  \
0            Comedy            Comedy          73       230666     False   
1  Ghost (Acoustic)  Ghost - Acoustic          55       149610     False   
2    To Begin Again    To Begin Again          57       210826     False   

   danceability  energy  ...  loudness  mode  speechiness  acousticness  \
0         0.676   0.461  ...    -6.746     0       0.1430        0.0322   
1         0.420   0.166  ...   -17.235     1       0.0763        0.9240   
2         0.438   0.359  ...    -9.734     1       0.0557        0.2100   

   instrumentalness  

In [ ]:
# 1. Install necessary library for Collaborative Filtering
!pip install scikit-surprise -q

# 2. Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from surprise import Dataset, Reader, KNNBasic

# ==========================================
#  YAHAN APNI FILE KA NAAM LIKHEIN
file_name = '/content/sample_data/dataset.csv' # Agar file ka naam kuch aur hai, toh yahan change karein
# ==========================================

try:
    df = pd.read_csv(file_name)
    print(" Dataset successfully loaded!\n")
except FileNotFoundError:
    print(f" Error: '{file_name}' nahi mili. Tafseelan check karein ke file upload ho gayi hai.")
    df = None

if df is not None:
    # ==========================================
    # DATA PREPROCESSING (As per Proposal)
    # ==========================================
    print(" Data Preprocessing shuru ho rahi hai...")

    # Missing values drop karna
    df = df.dropna()

    # Agar dataset mein Like/Skip (1/0) nahi hai, toh demo ke liye generate karein
    if 'target' not in df.columns:
        df['target'] = np.random.choice([0, 1], size=len(df)) # 1 = Like, 0 = Skip

    # Agar User ID nahi hai (Collaborative filtering ke liye zaroori hai)
    if 'user_id' not in df.columns:
        df['user_id'] = np.random.randint(1, 100, size=len(df)) # 100 dummy users

    # Agar track/song ID nahi hai
    if 'track_id' not in df.columns:
        df['track_id'] = df.index

    # Features to encode and scale
    categorical_cols = [col for col in ['genre', 'track_genre', 'artist_name', 'artists'] if col in df.columns]
    continuous_cols = [col for col in ['tempo', 'loudness', 'danceability', 'energy', 'valence', 'popularity'] if col in df.columns]

    # Label Encoding (Artists, Genres)
    le = LabelEncoder()
    for col in categorical_cols:
        df[col] = le.fit_transform(df[col].astype(str))

    # Normalization / Scaling (Tempo, Loudness etc.)
    scaler = StandardScaler()
    if continuous_cols:
        df[continuous_cols] = scaler.fit_transform(df[continuous_cols])

    # Select Features (X) and Target (y)
    X = df[continuous_cols + categorical_cols]
    y = df['target']

    # Train/Test Split (80% Train, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    print(" Preprocessing Done. Data splitted (80% Train, 20% Test).\n")

    # ==========================================
    # SUPERVISED LEARNING (Classification Models)
    # ==========================================
    print("🚀 Training Classification Models...\n")

    models = {
        "Logistic Regression": LogisticRegression(max_iter=1000),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
    }

    results = {}

    for name, model in models.items():
        # Train model
        model.fit(X_train, y_train)

        # Predictions
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else [0]*len(y_test)

        # Calculate Metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        roc = roc_auc_score(y_test, y_prob) if sum(y_test) > 0 else 0

        results[name] = [acc, prec, rec, f1, roc]

        print(f"--- {name} ---")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall:    {rec:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        print(f"ROC-AUC:   {roc:.4f}\n")

    # ==========================================
    # COLLABORATIVE FILTERING
    # ==========================================
    print(" Building Collaborative Filtering Model (User Similarity)...")

    # Surprise library ko data is format mein chahiye: (user_id, item_id, rating)
    # Filter for positive interactions (target == 1) to avoid ZeroDivisionError in cosine similarity
    cf_data = df[df['target'] == 1][['user_id', 'track_id', 'target']]
    reader = Reader(rating_scale=(0, 1))
    dataset_surprise = Dataset.load_from_df(cf_data, reader)

    trainset = dataset_surprise.build_full_trainset()

    # Using Cosine Similarity with KNNBasic
    sim_options = {
        'name': 'cosine',
        'user_based': True  # Compute similarities between users
    }
    algo = KNNBasic(sim_options=sim_options, verbose=False)

    algo.fit(trainset)
    print(" Collaborative Filtering Model Trained Successfully!\n")

    # ==========================================
    # HYBRID SYSTEM DEMO
    # ==========================================
    print(" Hybrid Prediction Demo:")
    sample_user = df['user_id'].iloc[0]
    sample_song = df['track_id'].iloc[0]

    # 1. Prediction from Collaborative Filtering
    # Need to handle cases where a user/item might not be in the CF training set due to filtering
    if sample_user in cf_data['user_id'].unique() and sample_song in cf_data['track_id'].unique():
        cf_prediction = algo.predict(sample_user, sample_song).est
    else:
        cf_prediction = 0.5 # Default or neutral prediction if not in CF data

    # 2. Prediction from Supervised Learning (Random Forest)
    sample_features = X.iloc[[0]]
    rf_prediction = models["Random Forest"].predict_proba(sample_features)[0][1]

    # 3. Hybrid Score (Average of both)
    hybrid_score = (cf_prediction + rf_prediction) / 2

    print(f"User ID: {sample_user} | Song ID: {sample_song}")
    print(f"Collaborative Score: {cf_prediction:.2f}")
    print(f"Content-Based (RF) Score: {rf_prediction:.2f}")
    print(f" Final Hybrid Recommendation Score: {hybrid_score:.2f} (Close to 1 means LIKE)")
    print("\n Project Execution Complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 50.1 MB/s eta 0:00:00
 Dataset successfully loaded!

 Data Preprocessing shuru ho rahi hai...
 Preprocessing Done. Data splitted (80% Train, 20% Test).

🚀 Training Classification Models...

--- Logistic Regression ---
Accuracy:  0.5006
Precision: 0.4993
Recall:    0.7479
F1-Score:  0.5988
ROC-AUC:   0.5026

--- Random Forest ---
Accuracy:  0.4973
Precision: 0.4957
Recall:    0.4967
F1-Score:  0.4962
ROC-AUC:   0.4954

 Building Collaborative Filtering Model (User Similarity)...
 Collaborative Filtering Model Trained Successfully!

 Hybrid Prediction Demo:
User ID: 94 | Song ID: 5SuOikwiRyPMVoIQDJUgSV
Collaborative Score: 0.50
Content-Based (RF) Score: 0.15
 Final Hybrid Recommendation Score: 0.33 (Close to 1 means LIKE)

 Project Execution Complete!
